In [2]:
import os
import urllib.request
from urllib.parse import quote

# 1. 랭체인 필수 모듈 임포트
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# ----------------------------------------------------
# 2. 실습용 PDF 다운로드
# ----------------------------------------------------
# URL 인코딩 후 파일 다운로드 (한글 파일명 처리)
pdf_url = quote("2023_북한인권보고서.pdf", safe=":/")  # 실제 호스팅 URL 또는 로컬 경로
pdf_filename = "2023_북한인권보고서.pdf"

# 로컬에 파일이 없을 경우 다운로드 (로컬 파일이 이미 있다면 생략 가능)
if not os.path.exists(pdf_filename):
    # urllib.request.urlretrieve("다운로드_URL_경로/" + pdf_url, filename=pdf_filename)
    pass

# ----------------------------------------------------
# 3. PDF 로드 (페이지 단위 1차 로드)
# ----------------------------------------------------
# PyPDFLoader는 페이지 단위로 문서를 분할하여 Document 객체 리스트를 생성합니다.
loader = PyPDFLoader(pdf_filename)
pages = loader.load_and_split()
print(f"[1단계] PDF 로드 완료: 총 {len(pages)}개 페이지 청크")

# ----------------------------------------------------
# 4. 청크 2차 세분화 (RecursiveCharacterTextSplitter)
# ----------------------------------------------------
# 한 페이지 분량은 LLM 및 임베딩 입력으로 길 수 있으므로 1,000자 단위로 분할합니다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=0
)

# 주의: pages는 문자열이 아닌 Document 객체 리스트이므로 split_documents()를 사용해야
# 원본 메타데이터(source, page 등)가 그대로 유지됩니다.
splitted_docs = text_splitter.split_documents(pages)
print(f"[2단계] 2차 분할 완료: 총 {len(splitted_docs)}개 청크 생성")

# ----------------------------------------------------
# 5. Chroma DB 적재 및 로컬 영구 저장 (Persistence)
# ----------------------------------------------------
# 환경변수에 OPENAI_API_KEY가 등록되어 있어야 합니다.
embeddings = OpenAIEmbeddings()

persist_dir = "./chroma_test_db"

# documents를 벡터화하여 로컬 디렉터리(persist_directory)에 SQLite/Parquet 파일로 영구 저장
db = Chroma.from_documents(
    documents=splitted_docs,
    embedding=embeddings,
    persist_directory=persist_dir
)

# 적재된 총 문서 청크 개수 확인
print(f"[3단계] Chroma DB 적재 완료 - 총 문서 수: {db._collection.count()}")

# ----------------------------------------------------
# 6. 저장된 로컬 파일로부터 Chroma DB 복원 (Reload)
# ----------------------------------------------------
# 새로운 세션이나 프로그램 재실행 시, 기존 저장 경로에서 DB 객체를 즉시 로드합니다.
db_from_file = Chroma(
    persist_directory=persist_dir,
    embedding_function=embeddings
)
print(f"[4단계] 로컬 파일에서 복원 완료 - 문서 수: {db_from_file._collection.count()}")

# ----------------------------------------------------
# 7. 유사도 검색 (Similarity Search)
# ----------------------------------------------------
question = "북한의 교육 과정"

# 1) 기본 유사도 검색 (기본 k=4)
search_results = db_from_file.similarity_search(question)
print(f"\n[5단계] 기본 검색 결과: 총 {len(search_results)}건 발견")
print("-" * 50)
print(f"가장 유사한 청크 본문 미리보기:\n{search_results[0].page_content[:200]}...")
print(f"출처 메타데이터: {search_results[0].metadata}")
print("-" * 50)

# 2) 상위 k개 제한 및 유사도 점수(Relevance Score) 함께 확인
top_k_docs = db_from_file.similarity_search_with_relevance_scores(question, k=3)
print("\n[6단계] 상위 3개 청크 및 유사도 점수:")
for i, (doc, score) in enumerate(top_k_docs, 1):
    page_num = doc.metadata.get("page", "알수없음")
    print(f"\n[{i}] 유사도 점수: {score:.4f} | 페이지: {page_num}")
    print(f"본문 요약: {doc.page_content[:150].strip()}...")

[1단계] PDF 로드 완료: 총 445개 페이지 청크
[2단계] 2차 분할 완료: 총 497개 청크 생성


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
C:\Users\HKIT\AppData\Local\Temp\ipykernel_13164\2616276331.py:67: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  db_from_file = Chroma(
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


[3단계] Chroma DB 적재 완료 - 총 문서 수: 497
[4단계] 로컬 파일에서 복원 완료 - 문서 수: 497


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



[5단계] 기본 검색 결과: 총 4건 발견
--------------------------------------------------
가장 유사한 청크 본문 미리보기:
2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에게 충분히 제공되지 않고 학년을 마치면 다음 학
년에 교과서를 물려주어야 했다는 사례가 다수 수집되었다. 소학교
부터 학교운영비, 꼬마계획 등의 비용을 내야했다는 진술이 꾸준히 
...
출처 메타데이터: {'creationdate': '2023-07-31T13:50:27+09:00', 'creator': 'Adobe InDesign CS6 (Windows)', 'moddate': '2023-07-31T13:57:54+09:00', 'page': 41, 'page_label': '42', 'producer': 'Adobe PDF Library 10.0.1', 'source': '2023_북한인권보고서.pdf', 'total_pages': 448, 'trapped': '/False'}
--------------------------------------------------

[6단계] 상위 3개 청크 및 유사도 점수:

[1] 유사도 점수: 0.8285 | 페이지: 41
본문 요약: 2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에게 충분히 제공되지 않고 학년을 마치면 다음 학
년에 교과서를 물려주어야 했다는 사례가 다...

[2] 유사도 점수: 0.8282 | 페이지: 283
본문 요약: 309		북한의	학제는	2012년	전반적	의무교육(유치원	1년,	소학교	5년,	초급중학교	3년,	고급중학교	3년)으로	
개편되었는데,	학제개편	이전에는	초급중학교와	고급중학교를	통합하여	중학교	6년	과정

In [2]:
import os
import urllib.request
from urllib.parse import quote

# 1. 필수 라이브러리 임포트
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# ----------------------------------------------------
# 2. PDF 로드 
# ----------------------------------------------------
pdf_filename = "2023_북한인권보고서.pdf"

loader = PyPDFLoader(pdf_filename)
pages = loader.load_and_split()
print(f"[1단계] PDF 로드 완료: 총 {len(pages)}개 페이지 청크")

# ----------------------------------------------------
# 3. 청크 2차 분할 
# ----------------------------------------------------

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=0
)

# Document 객체 리스트의 메타데이터(출처, 페이지)를 유지하며 분할
splitted_docs = text_splitter.split_documents(pages)
print(f"[2단계] 2차 분할 완료: 총 {len(splitted_docs)}개 청크 생성")

# ----------------------------------------------------
# 4. FAISS 벡터 데이터베이스 적재
# ----------------------------------------------------
# 환경변수 OPENAI_API_KEY 사용
embeddings = OpenAIEmbeddings()

# FAISS 인덱스 생성 및 문서 적재
faiss_db = FAISS.from_documents(
    documents=splitted_docs, 
    embedding=embeddings
)
# FAISS는 index.ntotal 속성으로 적재된 벡터(청크) 수를 확인합니다.
print(f"[3단계] FAISS 적재 완료 - 총 문서 수: {faiss_db.index.ntotal}")

# ----------------------------------------------------
# 5. FAISS 로컬 저장 및 다시 로드 
# ----------------------------------------------------
# 로컬 디렉터리에 인덱스(index.faiss) 및 메타데이터 매핑(index.pkl) 저장
save_dir = "faiss_index"
faiss_db.save_local(save_dir)
print(f"[4단계] '{save_dir}' 디렉터리에 로컬 저장 완료")

# 로컬 파일로부터 FAISS 복원
# (pickle 역직렬화 보안 동의 옵션 필수 지정)
new_db_faiss = FAISS.load_local(
    save_dir, 
    embeddings, 
    allow_dangerous_deserialization=True
)
print(f"[5단계] 로컬 파일에서 복원 완료 - 총 문서 수: {new_db_faiss.index.ntotal}")

# ----------------------------------------------------
# 6. 유사도 검색 및 상위 결과 출력
# ----------------------------------------------------
question = "북한의 교육 과정"

# 유사도 검색 실행
docs = new_db_faiss.similarity_search(question)
print(f"\n[6단계] 검색 완료 - 검색된 연관 문서 수: {len(docs)}")
print("=" * 60)

# 검색 결과 순회 출력 
for i, doc in enumerate(docs, 1):
    page_num = doc.metadata.get("page", "정보 없음")
    print(f"[검색 결과 {i}] (원본 페이지: {page_num})")
    print(doc.page_content.strip()[:200] + "...")
    print("-" * 60)


# ### 슬라이드 기준 주요 동작 포인트
# 1. **문서 수 확인 방식 차이**:
#    Chroma는 `db._collection.count()`를 사용하지만, FAISS는 **`faiss_db.index.ntotal`**로 총 적재 청크 수를 확인합니다[cite: 12, 15].
# 2. **역직렬화 보안 옵션**:
#    FAISS가 저장한 본문 메타데이터 매핑 파일(`index.pkl`)을 불러올 때는 반드시 **`allow_dangerous_deserialization=True`**를 명시해야 로드됩니다[cite: 13, 14, 15].
# 3. **C++ 충돌 방지**:
#    FAISS는 SQLite나 내부 C++ 스레드 락 문제를 일으키지 않아 Windows Jupyter 환경에서 커널 종료(Crash) 없이 안정적으로 실행됩니다.

[1단계] PDF 로드 완료: 총 445개 페이지 청크
[2단계] 2차 분할 완료: 총 497개 청크 생성
[3단계] FAISS 적재 완료 - 총 문서 수: 497
[4단계] 'faiss_index' 디렉터리에 로컬 저장 완료
[5단계] 로컬 파일에서 복원 완료 - 총 문서 수: 497

[6단계] 검색 완료 - 검색된 연관 문서 수: 4
[검색 결과 1] (원본 페이지: 41)
2023 북한인권보고서
40
명목의 교육비용이 전가되고 있는 것으로 나타났다. 교과서는 ‘교과
서 요금’이라는 명목으로 일정 금액을 내야하는 경우가 많으며, 교
과서가 모든 학생에게 충분히 제공되지 않고 학년을 마치면 다음 학
년에 교과서를 물려주어야 했다는 사례가 다수 수집되었다. 소학교
부터 학교운영비, 꼬마계획 등의 비용을 내야했다는 진술이 꾸준히 
...
------------------------------------------------------------
[검색 결과 2] (원본 페이지: 283)
309		북한의	학제는	2012년	전반적	의무교육(유치원	1년,	소학교	5년,	초급중학교	3년,	고급중학교	3년)으로	
개편되었는데,	학제개편	이전에는	초급중학교와	고급중학교를	통합하여	중학교	6년	과정(1972년~2011
년)으로	운영하였고,	중학교	또는	고등중학교라고	칭하였다.(통일부	국립통일교육원,	『북한의	이해』,	
2022)...
------------------------------------------------------------
[검색 결과 3] (원본 페이지: 185)
2023 북한인권보고서
184
데, 당국이 실시하는 반종교 교육을 통해 기독교를 접한 경우였다. 기
독교 관련 북한당국의 반종교 교육은 학교 교과과정에서 뿐만 아니
라 졸업 후 조직생활을 통해서도 이루어지고 있었다. 수집된 증언에 
따르면 북한에서 반종교 교육을 받고 종교에 대한 부정적 인식이 증
가했다고 한다. 기독교를 믿는 사람을 반동분자로 인식하고 있...
-----------------